# Lesson 10 · Monads & Algebras

Every adjunction, folded in half, becomes a monad: an endofunctor that knows how to flatten itself. Its laws are the reason effects compose, and its algebras are the structures it describes — beginning with the monoids of Lesson 1.

This notebook collects every Python block from the optional labs and exercises of [the lesson page](../lesson-10-monads-and-algebras.html), in page order. Run the cells from top to bottom; later cells use names defined in earlier ones.

Python is optional in this course — the lesson is complete without it. *Generated by `tools/sync_code.py`: edit the lesson page, not this file.*

## §7 — Monads by hand

**`monad_laws.py`** — List and Maybe, laws by enumeration

In [1]:
from itertools import product as cartesian

eta  = lambda x: [x]
mu   = lambda xss: [x for xs in xss for x in xs]     # Lesson 3's flatten
fmap = lambda f, xs: [f(x) for x in xs]

def lists_over(base, maxlen=2):
    out = [[]]
    for k in range(1, maxlen + 1):
        out += [list(t) for t in cartesian(base, repeat=k)]
    return out

T1 = lists_over([0, 1]); T2 = lists_over(T1); T3 = lists_over(T2)

assert all(mu(mu(x)) == mu(fmap(mu, x)) for x in T3)          # associativity
assert all(mu(eta(x)) == x == mu(fmap(eta, x)) for x in T1)   # unit laws
print(f"List : associativity on {len(T3)} T\u00b3 values, units on {len(T1)} — pass")

J     = lambda x: ('Just', x)                        # Maybe's eta
fmapM = lambda f, m: None if m is None else J(f(m[1]))
muM   = lambda mm: None if mm is None else mm[1]     # collapse one layer

M1 = [None] + [J(x) for x in (0, 1)]
M2 = [None] + [J(m) for m in M1]
M3 = [None] + [J(m) for m in M2]

assert all(muM(muM(m)) == muM(fmapM(muM, m)) for m in M3)
assert all(muM(J(m)) == m == muM(fmapM(J, m)) for m in M1)
print(f"Maybe: associativity on {len(M3)} T\u00b3 values, units on {len(M1)} — pass")

List : associativity on 3307 T³ values, units on 7 — pass
Maybe: associativity on 5 T³ values, units on 3 — pass


**`powerset_monad.py`** — unions of unions

In [2]:
from itertools import combinations

def powerset(xs):
    xs = list(xs)
    return [frozenset(c) for r in range(len(xs) + 1)
            for c in combinations(xs, r)]

etaP  = lambda x: frozenset([x])                        # singleton
muP   = lambda ss: frozenset(x for s in ss for x in s)  # union
fmapP = lambda f, s: frozenset(f(x) for x in s)         # direct image

P1 = powerset([0, 1]); P2 = powerset(P1); P3 = powerset(P2)
assert all(muP(muP(a)) == muP(fmapP(muP, a)) for a in P3)
assert all(muP(etaP(s)) == s == muP(fmapP(etaP, s)) for s in P1)
print(f"Powerset: associativity on {len(P3)} values,",
      f"units on {len(P1)} — pass")

Powerset: associativity on 65536 values, units on 4 — pass


**`kleisli_rel.py`** — relations are Kleisli arrows

In [3]:
def kleisli(g, f):
    "g after f, for the power set monad"
    return lambda a: muP(fmapP(g, f(a)))

def compose_rel(S, R):
    "S after R, for relations given as sets of pairs"
    return {(a, c) for (a, b) in R for (b2, c) in S if b == b2}

R = {(1, 'a'), (1, 'b'), (2, 'c')}
S = {('a', 'x'), ('b', 'y'), ('c', 'y')}
f = lambda a: frozenset(b for (a2, b) in R if a2 == a)
g = lambda b: frozenset(c for (b2, c) in S if b2 == b)

graph = {(a, c) for a in (1, 2) for c in kleisli(g, f)(a)}
print("Kleisli composite   :", sorted(graph))
print("relational composite:", sorted(compose_rel(S, R)))

Kleisli composite   : [(1, 'x'), (1, 'y'), (2, 'y')]
relational composite: [(1, 'x'), (1, 'y'), (2, 'y')]


**`evaluators.py`** — the sum is an algebra, the average is not

In [4]:
from fractions import Fraction

avg = lambda xs: Fraction(sum(xs), len(xs))       # an exact average
T2 = [xss for xss in lists_over(lists_over([0, 1]))
      if xss and all(xss)]            # non-empty lists of non-empty lists
ok_sum = all(sum(mu(xss)) == sum(fmap(sum, xss)) for xss in T2)
bad    = [xss for xss in T2 if avg(mu(xss)) != avg(fmap(avg, xss))]
print("sum satisfies the algebra law on all", len(T2), "values:", ok_sum)
print("the average fails it on", len(bad), "of them; first:", bad[0])

sum satisfies the algebra law on all 42 values: True
the average fails it on 12 of them; first: [[0], [0, 1]]


## Exercise 4

**`writer.py`**

In [5]:
etaW  = lambda x: (x, "")
fmapW = lambda f, p: (f(p[0]), p[1])
def muW(pp):
    (x, inner), outer = pp
    return (x, outer + inner)

logs = ["", "a", "bc"]
W1 = [(x, w) for x in (0, 1) for w in logs]
W2 = [(p, w) for p in W1 for w in logs]
W3 = [(q, w) for q in W2 for w in logs]

assert all(muW(muW(q)) == muW(fmapW(muW, q)) for q in W3)
assert all(muW(etaW(p)) == p == muW(fmapW(etaW, p)) for p in W1)
print(f"Writer: associativity on {len(W3)} values, units on {len(W1)} — pass")

Writer: associativity on 54 values, units on 6 — pass


## Exercise 5

**`kleisli_list.py`**

In [6]:
kleisli = lambda g, f: (lambda x: mu(fmap(g, f(x))))
f = lambda x: [x, x + 1]
g = lambda x: [x * 2]
h = lambda x: [x, -x]

assert all(kleisli(kleisli(h, g), f)(x) == kleisli(h, kleisli(g, f))(x)
           for x in range(-5, 6))
assert all(kleisli(f, eta)(x) == f(x) == kleisli(eta, f)(x)
           for x in range(-5, 6))
print("laws pass;  ((h .K g) .K f)(2) =", kleisli(kleisli(h, g), f)(2))

laws pass;  ((h .K g) .K f)(2) = [4, -4, 6, -6]


## Exercise 7

**`algebra.py`**

In [7]:
a = sum                          # a : List Z -> Z,  with a([]) == 0

assert all(a(eta(x)) == x for x in (0, 1, 5, -3))          # a . eta = id
T2 = lists_over(lists_over([0, 1]))
assert all(a(mu(xss)) == a(fmap(a, xss)) for xss in T2)    # a . mu = a . Ta
print(f"sum is a List-algebra: laws pass on {len(T2)} T² values")

sum is a List-algebra: laws pass on 57 T² values
